# Ground Truth problems

These are problems for which the data generating process is a known model, 

$$ y = \phi^*(\mathbf{x}, \theta^*) $$

We assess how well symbolic regression algorithms find the form of the model, $\phi^*$, with some leniency on $\theta^*$ (we allow the model to be off by a constant or a scalar). 

In [ ]:
import pandas as pd
import json
import numpy as np
from glob import glob
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rc('pdf', fonttype=42)
import os
sns.set(font_scale=1.2)

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Nimbus Roman', 'Times New Roman', 'Liberation Serif', 'DejaVu Serif'],
})

rdir  = '../../results/ground-truth/'
figdir = f'../figs/ground-truth/'
datadir = '../../datasets/rethinking_feynman/'

print('figdir:',figdir)
def save(h=None,name='tmp',pad=1.08,w_pad=None):
    name = name.strip().replace(' ','-').replace('%','pct')
    if h == None:
        h = plt.gcf()
    plt.tight_layout(pad=pad, w_pad=w_pad)
    print('saving',name+'.pdf')
    if not os.path.exists(figdir):
        os.makedirs(figdir)
    plt.savefig(figdir+'/'+name+'.pdf', dpi=400, bbox_inches='tight')

# read data from feather

In [ ]:
df_results = pd.read_feather(rdir+'results.feather')

# results.feather already includes the NeurIPS 2021 (v2.0) baselines plus runs merged via merge_groundtruth_results.py

print('mean trial count:')
print(df_results.groupby('algorithm')['dataset'].count().sort_values()
      / df_results.dataset.nunique())


# exclude Feynman datasets only RASR ran on

Some Feynman problems only have RASR results (`rasr_94m`) and none from the
baseline methods, so they can't support a cross-method comparison. Drop them.

In [ ]:
RASR_ALGS = {
    'rasr_94m',
}

non_rasr_datasets = set(
    df_results.loc[~df_results['algorithm'].isin(RASR_ALGS), 'dataset'].unique()
)
feynman_datasets = set(
    df_results.loc[df_results['data_group'] == 'Feynman', 'dataset'].unique()
)
rasr_only_datasets = sorted(feynman_datasets - non_rasr_datasets)

print(f'excluding {len(rasr_only_datasets)} Feynman datasets with only RASR results:')
print(rasr_only_datasets)

df_results = df_results[~df_results['dataset'].isin(rasr_only_datasets)].reset_index(drop=True)

# fill in missing seeds and missing datasets

Not every method finished all 10 random-seed trials on every dataset/noise
combination, and a couple of methods never ran on some datasets at all.
Noise levels a method has zero results for anywhere are left alone entirely
(that noise level was simply never run for that method, so there's nothing
to fill in). For the noise levels a method *does* have results for:

- **Missing seeds**: if a (algorithm, dataset, target_noise) group has at
  least one completed seed but fewer than the full 10, add rows for the
  missing seeds with `r2_test = 0` and `model_size` set to the median
  expression size of that group's completed seeds.
- **Missing dataset**: if a method has zero results for an entire dataset
  (across all seeds) but did run on other datasets, add rows for all 10
  seeds with `r2_test = 0`. There's no in-dataset data to take a median
  from here, so `model_size` is set to that method's mean expression size
  across all of its actual (non-fabricated) results.

In [ ]:
# stats from the ORIGINAL data, so the two fill steps don't feed off each other
ALL_SEEDS = sorted(df_results['random_state'].unique())
ALL_DATASETS = set(df_results['dataset'].unique())
ALG_DATASETS = df_results.groupby('algorithm')['dataset'].apply(lambda x: set(x.unique())).to_dict()
ALG_ACTIVE_NOISES = df_results.groupby('algorithm')['target_noise'].apply(lambda x: set(x.unique())).to_dict()
ALG_MEAN_SIZE = df_results.groupby('algorithm')['model_size'].mean().to_dict()

# columns without a sensible fabricated value stay NaN; mse_test/mae_test keep the template row's values
# (they feed integer rank columns)
_NA_COLS = [
    'process_time', 'training time (s)', 'training time (hr)', 'grid_time',
    'mse_train', 'mae_train',
    'symbolic_model', 'simplified_symbolic_model', 'simplified_complexity',
    'symbolic_error', 'symbolic_fraction',
    'symbolic_error_is_zero', 'symbolic_error_is_constant', 'symbolic_fraction_is_constant',
    'sympy_exception',
]

# missing seeds: groups with at least one but fewer than the full 10 seeds
fill_rows = []
for (algorithm, dataset, target_noise), g in df_results.groupby(
        ['algorithm', 'dataset', 'target_noise']):
    missing_seeds = [s for s in ALL_SEEDS if s not in set(g['random_state'])]
    if not missing_seeds:
        continue
    median_size = g['model_size'].median()
    template = g.iloc[0]
    for seed in missing_seeds:
        row = template.copy()
        row['random_state'] = seed
        row['r2_test'] = 0.0
        row['r2_train'] = 0.0
        row['r2_zero_test'] = 0.0
        row['model_size'] = median_size
        row['symbolic_solution'] = False
        for col in _NA_COLS:
            row[col] = np.nan
        fill_rows.append(row)

if fill_rows:
    df_fill = pd.DataFrame(fill_rows)
    print(f'adding {len(df_fill)} rows for missing seeds '
          f'(r2_test=0, model_size=median of completed seeds)')
    df_results = pd.concat([df_results, df_fill], ignore_index=True)

In [ ]:
# missing datasets: no results for a dataset (all seeds, all noise levels) but results on others; only
# noise levels the method ran elsewhere (ALG_ACTIVE_NOISES) are filled
missing_ds_rows = []
for algorithm, datasets_present in ALG_DATASETS.items():
    missing_datasets = ALL_DATASETS - datasets_present
    if not missing_datasets:
        continue
    active_noises = ALG_ACTIVE_NOISES[algorithm]
    mean_size = ALG_MEAN_SIZE[algorithm]
    alg_template = df_results[df_results['algorithm'] == algorithm].iloc[0]
    for dataset in missing_datasets:
        ds_template = df_results[df_results['dataset'] == dataset].iloc[0]
        for noise in active_noises:
            for seed in ALL_SEEDS:
                row = ds_template.copy()
                row['algorithm'] = algorithm
                for col in ['symbolic_alg', 'gp_alg', 'pre_trained', 'complexity_function']:
                    row[col] = alg_template[col]
                row['target_noise'] = noise
                row['random_state'] = seed
                row['r2_test'] = 0.0
                row['r2_train'] = 0.0
                row['r2_zero_test'] = 0.0
                row['model_size'] = mean_size
                row['symbolic_solution'] = False
                for col in _NA_COLS:
                    row[col] = np.nan
                missing_ds_rows.append(row)

if missing_ds_rows:
    df_fill2 = pd.DataFrame(missing_ds_rows)
    print(f'adding {len(df_fill2)} rows for methods missing an entire dataset '
          f'(r2_test=0, model_size=method mean)')
    df_results = pd.concat([df_results, df_fill2], ignore_index=True)

# compute symbolic solutions

In [ ]:
# df_results.loc[:,'symbolic_solution'] = df_results[['symbolic_error_is_zero',
#                                                     'symbolic_error_is_constant',
#                                                     'symbolic_fraction_is_constant']
#                                                    ].apply(any,raw=True, axis=1)
# # clean up any corner cases (constant models, failures)
# df_results.loc[:,'symbolic_solution'] = df_results['symbolic_solution'] & ~df_results['simplified_symbolic_model'].isna() 
# df_results.loc[:,'symbolic_solution'] = df_results['symbolic_solution'] & ~(df_results['simplified_symbolic_model'] == '0')
# df_results.loc[:,'symbolic_solution'] = df_results['symbolic_solution'] & ~(df_results['simplified_symbolic_model'] == 'nan')

# # save results for detailed tabulating
# df_results.to_feather(rdir+'ground-truth_solns.feather')

## summarize results by dataset, including ranking

In [ ]:
df_results2 = df_results.copy()
df_results2['algorithm_count'] = df_results2.groupby(['dataset','target_noise'])['algorithm'].transform('nunique')
df_results2['random_state_repeats'] = df_results2.groupby(['algorithm','dataset','target_noise'])['random_state'].transform('nunique')

df_results2['accuracy_solution'] = df_results2['r2_test'].apply(lambda x: x > 0.999).astype(float)

for soln in ['accuracy_solution','symbolic_solution']:
    df_results2[soln+'_rate'] = df_results2.groupby(['algorithm','dataset','target_noise'])[soln].transform('mean')
                                       
for col in [c for c in df_results2.columns if c.endswith('test') or c.endswith('size')]:
    ascending = 'r2' not in col
    df_results2[col+'_rank_per_trial']=df_results2.groupby(
                        ['dataset','target_noise','random_state'])[col].transform(
                                                                              lambda x: 
                                                                              round(x,3).rank(
                                                                              ascending=ascending).astype(int))
    
df_sum = df_results2.groupby(['algorithm','dataset','target_noise','data_group'],as_index=False).median(numeric_only=True)
for col in [c for c in df_sum.columns if any([c.endswith(n) for n in ['test','size','rate']])]:
    ascending = 'r2' not in col and 'solution' not in col
    df_sum[col+'_rank']=df_sum.groupby(['dataset','target_noise'])[col].transform(
        lambda x:  round(x,3).rank(ascending=ascending).astype(int) )
    df_sum[col+'_norm'] = df_sum.groupby(['dataset','target_noise'])[col].transform(lambda x: (x-x.min())/(x.max()-x.min()))
for soln in ['accuracy_solution','symbolic_solution']:
    df_sum[soln +'_rate_(%)'] = df_sum[soln+'_rate'].apply(lambda x: x*100)
df_sum['rmse_test'] = df_sum['mse_test'].apply(np.sqrt)
df_sum['log_mse_test'] = df_sum['mse_test'].apply(lambda x: np.log(1+x))
df_results = df_results2
df_sum

# save summary data
used for statistical comparisons

In [ ]:
df_sum.to_csv(rdir+'/symbolic_dataset_results_sum.csv.gz',compression='gzip', index=False)

# solution rates by alg/dataset/noise

In [ ]:
df_sum.groupby(['target_noise','algorithm'])['symbolic_solution_rate_(%)'].mean().round(2).unstack().transpose()

# rename for display
Only affects the plots below: `results.feather`, the saved summary CSV, and the
solution-rate table above keep the raw algorithm names.

In [ ]:
_DISPLAY_RENAME = {
    'rasr_94m': 'RASR-94M-L',
    'FlashANSR': 'Flash-ANSR',
}
df_sum['algorithm'] = df_sum['algorithm'].replace(_DISPLAY_RENAME)


# plot comparisons

In [ ]:
sns.set_style('whitegrid')
# set_style resets font.family to sans-serif, so the paper font is applied after it.
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Nimbus Roman', 'Times New Roman', 'Liberation Serif', 'DejaVu Serif'],
    'mathtext.default': 'regular',
})
def compare(df_compare=None, x='r2_test',y='algorithm', row=None, col=None, scale=None, xlim=[], est=np.mean,
            orient='h', hue=None, **kwargs):
    df_compare = df_compare.copy()
    if row==None and col == None:
        aspect=1
    else:
        aspect=0.55
    tmp = df_compare.groupby(['target_noise',y])[x].apply(est).unstack().mean()
    order = tmp.sort_values(ascending=False).index
    
    for c in [x,y,row,col]:
        if c:
            df_compare = df_compare.rename(columns={c:c.replace('_',' ').title()})
        
    x = x.replace('_',' ').title()
    y = y.replace('_',' ').title()
    if row:
        row = row.replace('_',' ').title()
    if col:
        col = col.replace('_',' ').title()
    
    if scale=='log' and len(xlim)>0 and xlim[0] == 0:
        df_compare.loc[:,x] += 1
        xlim[0] = 1
        xnew = '1 + '+x
        df_compare=df_compare.rename(columns={x:xnew})
        x = xnew
    if orient=='v':
        tmp = x
        x = y
        y = tmp
    if col and not row:
        col_wrap = min(4, df_compare[col].nunique()) 
    else:
        col_wrap=None
        
    cat_args = dict(
                data=df_compare, 
                kind='point',
                y=y,
                x=x,
                order=order,
                row=row,
                col=col,
                col_wrap=col_wrap,
                margin_titles=True,
                aspect=aspect,
                hue=hue,
                legend_out=False,
    )
    cat_args.update(kwargs)
    g = sns.catplot( **cat_args )
    for ax in g.axes.flat: 
        ax.yaxis.grid(True)
        ax.set_ylabel('')
        ax.set_xlabel(ax.get_xlabel().replace('Symbolic ',''))
        if col:
            ttl = ax.get_title()
            ax.set_title(ttl.replace(col,'').replace('=',''))

    
    if len(xlim)>0:
        plt.xlim(xlim[0],xlim[1])
    if scale:
        plt.gca().set_xscale(scale)

    sns.despine(left=True, bottom=True)
    savename = '-'.join(['cat-'+cat_args['kind']+'plot',x+ '-by-'+ y])
    if row: savename += '_'+row
    if col: savename += '_'+col
    

In [ ]:
# Make the PairGrids: one per (data_group, primary metric), each with
# [primary metric, Expression Size] panels.

# 'median': median accuracy_solution over seeds (0.5 possible), like r2_test; 'mean': solve rate per dataset
ACCURACY_PER_DATASET_AGG = 'median'

accuracy_col = 'accuracy_solution_rate' if ACCURACY_PER_DATASET_AGG == 'mean' else 'accuracy_solution'
metrics = [
    ('r2_test', '$R^2$ Test'),
    (accuracy_col, 'Accuracy\n($R^2 > 0.999$)'),
]
extra_vars = ['model_size']
extra_titles = ['Expression\nSize']

noise_markers = ['x', 'o', 's', '+']
noise_marker_sizes = {'x': 5, 'o': 5, 's': 3.7, '+': 5}

_rc = {
    'font.size': 7, 'axes.titlesize': 7, 'axes.labelsize': 7,
    'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'legend.fontsize': 7, 'legend.title_fontsize': 7,
    'ytick.major.pad': 1.5,
}
for data_group in ['Feynman', 'Strogatz']:
    for metric, metric_title in metrics:
      with plt.rc_context(_rc):
          df_plot = df_sum[df_sum['data_group'] == data_group].copy()
          x_vars = [metric] + extra_vars
          titles = [metric_title] + extra_titles

          tmp = df_plot.groupby(['target_noise','algorithm'])[metric].mean().unstack().mean()
          order = tmp.sort_values(ascending=False).index

          # map noise levels to colours by rank; raw values would bunch 0/0.001/0.01 in similar shades
          noise_levels = sorted(df_plot['target_noise'].unique())
          noise_palette = dict(zip(noise_levels, sns.color_palette('flare_r', n_colors=len(noise_levels))))

          g = sns.PairGrid(df_plot,
                           x_vars=x_vars,
                           y_vars=['algorithm'],
                           height=2.3,
                           aspect=2.8 * 0.4656 / 2.3,
                           hue='target_noise',
                           hue_order=noise_levels,
                           palette=noise_palette,
                          )
          g.map(sns.pointplot,
                orient="h",
                order=order,
                markersize=5,
                markeredgewidth=0.95,
                err_kws={'linewidth': 1.15},
                capsize=0.15,
                join=False,
                estimator=np.mean,
                n_boot=1000,
                markers=noise_markers,
                alpha=0.9
               )

          for ax in g.axes.flat:
              for line in ax.lines:
                  marker_size = noise_marker_sizes.get(line.get_marker())
                  if marker_size is not None:
                      line.set_markersize(marker_size)

          plt.legend(title='Target Noise', loc='lower right')

          for ax, title in zip(g.axes.flat, titles):

              ax.set_xlabel('')
              ax.set_ylabel('')
              ax.set(title=title)

              if any([n in title.lower() for n in ['size','complexity','time']]):
                  ax.set_xscale('log')
              if title == '$R^2$ Test':
                  ax.set_xlim([0.5, 1])
              elif title == 'Mean Accuracy ($R^2 > 0.999$)':
                  ax.set_xlim([0, 1])

              ax.yaxis.grid(True)

          g.fig.canvas.draw()
          for label in g.axes.flat[0].get_yticklabels():
              if 'RASR' in label.get_text():
                  label.set_fontweight('bold')

          save(g, f'pairgrid_{data_group.lower()}_{metric}', pad=0.3, w_pad=0.3)
